# CART — Classification and Regression Trees (1984)

_Papers · Classical ML_

**Grow a decision tree by greedily picking, at each node, the feature-and-threshold split that most reduces a simple impurity score (Gini) — then prune it back with a cost-complexity penalty so it does not overfit.**

---

This notebook builds CART from scratch, one piece at a time. Run each cell, read the note above it, watch our tree reproduce scikit-learn's, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Define impurity and split scoring

CART measures how "mixed" a node's labels are with the **Gini impurity** `H = sum_k p_k(1 - p_k)`: zero when every example shares one label, larger when the classes are balanced. To score a candidate split we compute the *weighted-average* Gini of its two children — each child weighted by the fraction of rows it receives. A split is good when that weighted child impurity is low.

In [ ]:
import numpy as np

np.random.seed(0)

# ============================================================
# CART from scratch: Gini impurity, split score, greedy search.
# Criterion transcribed from scikit-learn's tree "Mathematical
# formulation" (it implements CART, Breiman et al. 1984).
# ============================================================

def gini(y):
    """Gini impurity  H = sum_k p_k (1 - p_k)  of a label vector."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return float(np.sum(p * (1.0 - p)))

def split_score(X, y, j, t):
    """Weighted-average child Gini for split (feature j <= threshold t).
       Returns None if a child would be empty."""
    left = X[:, j] <= t
    right = ~left
    nL, nR, n = left.sum(), right.sum(), len(y)
    if nL == 0 or nR == 0:
        return None
    return (nL / n) * gini(y[left]) + (nR / n) * gini(y[right])

### Step 2 — Search greedily for the best split

CART is greedy: at each node it tries **every feature** and **every candidate threshold**, then keeps the one split with the smallest weighted child Gini. The candidate thresholds are the midpoints between consecutive distinct feature values — any threshold between two values produces the same partition, so the midpoints cover all the distinct splits.

In [ ]:
def best_split(X, y):
    """Greedy CART split: try every feature and every midpoint threshold,
       keep the (feature, threshold, score) with the SMALLEST weighted Gini."""
    best = (None, None, np.inf)
    for j in range(X.shape[1]):
        vals = np.unique(X[:, j])
        cands = (vals[:-1] + vals[1:]) / 2.0          # midpoints between distinct values
        for t in cands:
            g = split_score(X, y, j, t)
            if g is not None and g < best[2] - 1e-12:
                best = (j, float(t), float(g))
    return best                                       # (feature, threshold, weighted Gini)


# --- WORKED EXAMPLE: assert every hand-computed number. ---
X1 = np.array([[1.0], [2.0], [3.0], [6.0], [7.0], [8.0]])
y1 = np.array([0, 0, 0, 1, 1, 1])
assert abs(gini(y1) - 0.5) < 1e-12                    # parent: 50/50 -> 0.5
assert abs(split_score(X1, y1, 0, 4.5) - 0.0)  < 1e-12  # perfect split -> 0
assert abs(gini(y1[X1[:, 0] > 2.5]) - 0.375)   < 1e-12  # right child {0,1,1,1} -> 0.375
assert abs(split_score(X1, y1, 0, 2.5) - 0.25) < 1e-12  # weighted -> 0.25

j, t, g = best_split(X1, y1)
assert j == 0 and abs(t - 4.5) < 1e-12 and abs(g - 0.0) < 1e-12
print("worked example OK:  parent gini=0.5  G(4.5)=0.0  G(2.5)=0.25  best=(feat 0, t=4.5, G=0.0)")

### Step 3 — Grow the tree recursively

A tree is just nested splits. `build` makes a leaf (predicting the majority label) and stops if it hit the depth limit or the node is already pure; otherwise it finds the best split, partitions the rows, and recurses on each side. `predict` then walks an example down the tree, going left when `x[j] <= t`, until it lands in a leaf.

In [ ]:
class Node:
    def __init__(self, **kw):
        self.__dict__.update(kw)

def build(X, y, depth, max_depth):
    node = Node(leaf=True, pred=int(np.round(y.mean())), gini=gini(y), n=len(y))
    if depth >= max_depth or gini(y) == 0.0 or len(np.unique(y)) == 1:
        return node
    j, t, g = best_split(X, y)
    if j is None:
        return node
    left = X[:, j] <= t
    node.leaf = False
    node.j = j
    node.t = t
    node.left  = build(X[left],  y[left],  depth + 1, max_depth)
    node.right = build(X[~left], y[~left], depth + 1, max_depth)
    return node

def predict(node, X):
    def one(nd, x):
        while not nd.leaf:
            nd = nd.left if x[nd.j] <= nd.t else nd.right
        return nd.pred
    return np.array([one(node, x) for x in X])

### Step 4 — Verify against scikit-learn's CART

On a toy two-blob dataset we grow our tree and compare it to scikit-learn's `DecisionTreeClassifier(criterion="gini")`, which implements the same CART algorithm. If our from-scratch code is correct, the root split (feature and threshold) and every prediction should match scikit-learn's exactly.

In [ ]:
# Toy two-blob dataset.
n = 60
Xa = np.random.randn(n // 2, 2) * 0.6                  # blob near (0, 0)    -> class 0
Xb = np.random.randn(n // 2, 2) * 0.6 + 2.2            # blob near (2.2,2.2) -> class 1
X = np.vstack([Xa, Xb])
y = np.array([0] * (n // 2) + [1] * (n // 2))

ours = build(X, y, 0, max_depth=3)
print("our root split: feature", ours.j, " threshold %.4f" % ours.t)

from sklearn.tree import DecisionTreeClassifier      # imported ONLY as the oracle

sk = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=0).fit(X, y)
print("sklearn root : feature", int(sk.tree_.feature[0]), " threshold %.4f" % float(sk.tree_.threshold[0]))

agree = (predict(ours, X) == sk.predict(X)).mean()
print("predictions agree with sklearn CART: %.4f" % agree)
assert agree == 1.0, "our CART must match scikit-learn's CART"
print("VERIFIED: from-scratch CART == scikit-learn DecisionTreeClassifier(gini).")
# Our small run: root = (feature 0, threshold ~1.1712) for both; predictions agree 1.0000.
# This is OUR toy run, not a number from the CART book.

## Visualize the data & results

_As CART is allowed to grow deeper, how fast does the weighted training impurity R(T) fall — the quantity cost-complexity pruning trades against tree size?_

### Step 1 — Rebuild the helpers and a leaf collector

This cell stands alone, so we re-define the same `gini` / `split_score` / `best_split` / `build` pieces from above, plus one new helper: `leaves` walks the tree and collects each leaf's row count and Gini. We will use those to compute the tree's overall training impurity.

In [ ]:
import numpy as np
np.random.seed(0)

def gini(y):
    if len(y) == 0:
        return 0.0
    _, c = np.unique(y, return_counts=True)
    p = c / c.sum()
    return float(np.sum(p * (1 - p)))

def split_score(X, y, j, t):
    L = X[:, j] <= t
    R = ~L
    nL, nR, n = L.sum(), R.sum(), len(y)
    if nL == 0 or nR == 0:
        return None
    return (nL / n) * gini(y[L]) + (nR / n) * gini(y[R])

def best_split(X, y):
    best = (None, None, np.inf)
    for j in range(X.shape[1]):
        v = np.unique(X[:, j])
        cs = (v[:-1] + v[1:]) / 2.0
        for t in cs:
            g = split_score(X, y, j, t)
            if g is not None and g < best[2] - 1e-12:
                best = (j, float(t), float(g))
    return best

class Node:
    def __init__(s, **k):
        s.__dict__.update(k)

def build(X, y, d, md):
    nd = Node(leaf=True, gini=gini(y), n=len(y))
    if d >= md or gini(y) == 0.0 or len(np.unique(y)) == 1:
        return nd
    j, t, g = best_split(X, y)
    if j is None:
        return nd
    L = X[:, j] <= t
    nd.leaf = False
    nd.left = build(X[L], y[L], d + 1, md)
    nd.right = build(X[~L], y[~L], d + 1, md)
    return nd

def leaves(nd, out):
    if nd.leaf:
        out.append((nd.n, nd.gini))
    else:
        leaves(nd.left, out)
        leaves(nd.right, out)

### Step 2 — Sweep the depth limit and measure R(T)

We rebuild the two-blob data, then grow the tree under depth limits 0 through 4. For each tree, `R(T)` is the row-weighted sum of leaf Gini values — the total training impurity. Watch it fall toward 0 as depth (and leaf count) grow: every extra split can only keep or lower training impurity. That monotonic drop is exactly what cost-complexity pruning balances against the number of leaves.

In [ ]:
n = 60
Xa = np.random.randn(n // 2, 2) * 0.6
Xb = np.random.randn(n // 2, 2) * 0.6 + 2.2
X = np.vstack([Xa, Xb])
y = np.array([0] * (n // 2) + [1] * (n // 2))

print("root split:", best_split(X, y)[:2])   # -> (0, ~1.1712)
for md_limit in range(5):
    lv = []
    leaves(build(X, y, 0, md_limit), lv)
    R = sum(ni * gi for ni, gi in lv) / len(y)
    print("max_depth=%d  R(T)=%.4f  n_leaves=%d" % (md_limit, R, len(lv)))
# Our small run -> R(T): 0.5000, 0.0323, 0.0000, 0.0000, 0.0000 (depths 0..4)
#               -> root split = (feature 0, threshold ~1.1712)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute a split score by hand. A node has labels $y = [0, 0, 1, 1, 1]$ and one feature
            $x = [1, 2, 3, 4, 5]$. Consider the threshold $t = 2.5$ (left = rows with $x \le 2.5$). Compute the
            parent Gini, each child's Gini, and the weighted split score $G$. Is this a perfect split?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Parent Gini: $p_0 = 2/5, p_1 = 3/5$, so $H = \tfrac25\cdot\tfrac35 + \tfrac35\cdot\tfrac25 = \tfrac{6}{25}+\tfrac{6}{25} = 0.48$. — _Gini sums $p_k(1-p_k)$ over classes; this is the impurity before any split._
- Left = $\{0,0\}$ (rows with $x \le 2.5$): pure, Gini $= 0$. Right = $\{1,1,1\}$: pure, Gini $= 0$. — _The threshold $2.5$ separates the two $0$s from the three $1$s exactly._
- $G = \tfrac{2}{5}(0) + \tfrac{3}{5}(0) = 0$. — _Each child weighted by its row fraction; both children are pure._

**Answer:** Parent Gini $= \mathbf{0.48}$; both children are pure (Gini $0$); weighted score
                 $G = \mathbf{0}$. Yes, it is a perfect split &mdash; impurity reduction
                 $0.48 - 0 = 0.48$, the largest possible for this node, because $t = 2.5$ cleanly separates the
                 two classes.

</details>

**Problem 2.** The pruning / overfitting ablation. You grow your from-scratch CART to ever-larger depth
            limits on the same training data and record the training impurity $R(T)$ and the number of leaves.
            What happens to $R(T)$ as depth grows, why does that not mean a deeper tree is always better,
            and what does the cost-complexity term $\alpha|\widetilde{T}|$ do about it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Increase max depth $0, 1, 2, 3, \dots$ and record $R(T)$ and the leaf count. — _Each extra split can only keep or lower training impurity, so $R(T)$ falls (toward $0$) as depth grows, while leaves multiply._
- Note that low training impurity is not the goal &mdash; test performance is. A tree grown to pure leaves fits noise. — _Driving $R(T)$ to $0$ memorizes the training set; that is overfitting, which hurts on unseen data._
- Add the penalty: minimize $R_\alpha(T) = R(T) + \alpha|\widetilde{T}|$ and pick $\alpha$ by cross-validation. — _The per-leaf fine makes a new split pay only if it lowers $R(T)$ by more than $\alpha$, so the tree stops growing past the point where splits stop helping._

**Answer:** $R(T)$ monotonically decreases toward $0$ as depth grows, because every split can only
                 keep or reduce training impurity. But minimal training impurity is overfitting &mdash; the tree
                 memorizes noise and generalizes worse. Cost-complexity pruning counters this: by adding
                 $\alpha|\widetilde{T}|$, each extra leaf must earn its keep (reduce $R(T)$ by more than
                 $\alpha$), and $\alpha$ is tuned by cross-validation to the size that generalizes best. The
                 CODEVIZ panel shows our $R(T)$ falling to $0$ as the depth limit rises &mdash; the curve pruning
                 trades against tree size.

</details>

**Problem 3.** Your best_split tries every feature and every threshold and keeps the smallest $G$. On
            a node with two features where feature 0 perfectly separates the classes and feature 1 does not, what
            should the search return, and why is the parent Gini irrelevant to the choice between two candidate
            splits?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For feature 0, the separating threshold gives two pure children, so $G = 0$. — _A perfect separation makes both child Ginis $0$, and any weighted average of zeros is $0$._
- For feature 1, no threshold makes both children pure, so its best $G \gt 0$. — _If a feature cannot separate the classes, at least one child stays mixed, keeping $G$ above $0$._
- Pick the smallest $G$ overall: feature 0's split ($G = 0$). The parent Gini $H(Q_m)$ is the same constant subtracted from every candidate's reduction. — _Since $H(Q_m)$ does not depend on $\theta$, comparing splits by minimum $G$ is identical to comparing by maximum reduction $H - G$._

**Answer:** The search returns the feature-0 split with $G = 0$ (two pure children), because it
                 has the smallest weighted child impurity of any candidate. The parent Gini $H(Q_m)$ is
                 irrelevant to the choice because it is a fixed constant for the node &mdash; it appears in
                 every candidate's reduction $H - G$, so ranking splits by minimum $G$ gives the same winner as
                 ranking by maximum impurity reduction.

</details>